# Router Level Category Classifier Notebook

This notebook is written for beginners.

What this notebook does now:
- reads the full `router_level_dataset.csv`
- explains the data step by step
- uses one combined dataset instead of the source seen/unseen split
- removes duplicate rows before modeling
- drops `record_id`, `app_name`, and `split` from the feature matrix
- balances categories before a stratified 80:20 train/test split
- tunes 5 lightweight single models
- compares 5 hard-voting hybrid models
- ends with one best lightweight model and one best hybrid model for later C conversion

What this notebook does not do:
- it does not rewrite the source CSV file
- it does not guarantee 90%+ test accuracy in advance
- it does not prove superiority over external products without a matched benchmark


## How To Use This Notebook

1. Run the install cell only if the import cell fails.
2. Then run the notebook from top to bottom.
3. Each code cell shows one small result.


## Step 1: Install `pandas` Only If Needed

If Step 2 works, you can skip this cell.

Run this cell only when `pandas` is not installed.


In [1]:
# Run the commented line only if Step 2 gives an import error in your own environment.
# %pip install pandas
print("Optional install cell skipped because pandas is already available in this environment.")


Optional install cell skipped because pandas is already available in this environment.


## Step 2: Import The Libraries We Need Now

For this first notebook, we only need `pandas` and a few built-in Python tools.


In [2]:
from io import StringIO
from pathlib import Path

import pandas as pd
from IPython.display import display

print("Libraries imported successfully.")


Libraries imported successfully.


## Step 3: Set Display Options

This makes notebook tables easier to read.


In [3]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

print("Display options are ready.")


Display options are ready.


## Step 4: Define The CSV File Path

We will read the full router-level dataset from the `outputs` folder.


In [4]:
dataset_path = Path("outputs") / "router_level_dataset.csv"
print("Dataset path:")
print(dataset_path.resolve())


Dataset path:
D:\New folder\combine\apps\Organized_by_App\outputs\router_level_dataset.csv


## Step 5: Confirm The File Exists

It is always good to check the file before reading it.


In [5]:
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {dataset_path}")

print("Dataset file found.")


Dataset file found.


## Step 6: Read The CSV Into A DataFrame

The dataframe name will be `df`.


In [6]:
df = pd.read_csv(dataset_path)
print("CSV loaded successfully into df.")


CSV loaded successfully into df.


## Step 7: Define The Main Target Column

For this dataset, the class label is `Category`.


In [7]:
target_col = "Category"
y = df[target_col]

print(f"Target column name: {target_col}")
print("First 10 target values:")
display(y.head(10).to_frame(name=target_col))


Target column name: Category
First 10 target values:


,Category
0,Adult
1,Adult
2,Adult
3,Adult
4,Adult
5,Adult
6,Adult
7,Adult
8,Adult
9,Adult


## Step 8: Quick Snapshot At One Glance

This gives a small summary table anyone can understand quickly.


In [8]:
quick_snapshot = pd.DataFrame(
    [
        {"item": "Rows", "value": f"{df.shape[0]:,}"},
        {"item": "Columns", "value": f"{df.shape[1]:,}"},
        {"item": "Unique apps", "value": f"{df['app_name'].nunique():,}"},
        {"item": "Unique categories", "value": f"{df[target_col].nunique():,}"},
        {"item": "Source split labels in CSV", "value": f"{df['split'].nunique():,}"},
    ]
)

display(quick_snapshot)


,item,value
0,Rows,"62,898"
1,Columns,117
2,Unique apps,200
3,Unique categories,4
4,Source split labels in CSV,2


## Step 9: Show The First 5 Rows

This is the easiest way to see what one sample looks like.


In [9]:
display(df.head())


,record_id,Category,app_name,split,transport_protocol,ip_version,protocol_family,flow_pattern_guess,traffic_type,traffic_type_confidence,traffic_type_is_guessed,app_behavior_hint,src_port,dst_port,is_tcp,is_udp,is_https,is_dns,is_http,duration_seconds,forward_duration_seconds,backward_duration_seconds,long_flow,orig_packets,resp_packets,total_packets,orig_bytes,resp_bytes,total_bytes,packets_per_second,bytes_per_second,bytes_per_packet,forward_packets_per_second,backward_packets_per_second,throughput_ratio,flow_density,upload_download_ratio,packet_ratio,byte_ratio,directional_packet_ratio,directional_byte_ratio,server_dominance,response_ratio,flow_symmetry,interaction_balance,flow_iat_mean_seconds,flow_iat_std_seconds,inter_arrival_variation,flow_iat_max_seconds,flow_iat_min_seconds,fwd_iat_mean_seconds,bwd_iat_mean_seconds,forward_iat_std_seconds,forward_iat_max_seconds,forward_iat_min_seconds,backward_iat_std_seconds,backward_iat_max_seconds,backward_iat_min_seconds,burst_score,burst_ratio,burst_intensity,time_variability_score,latency_score,forward_header_length,backward_header_length,pkt_len_mean,pkt_len_std,packet_size_variance,pkt_len_min,pkt_len_max,fwd_pkt_len_mean,bwd_pkt_len_mean,subflow_forward_packets,subflow_backward_packets,subflow_forward_bytes,subflow_backward_bytes,packet_variation_score,shape_score,active_mean_seconds,active_std_seconds,active_max_seconds,active_min_seconds,idle_mean_seconds,idle_std_seconds,idle_max_seconds,idle_min_seconds,activity_score,interaction_intensity,packet_intensity_score,byte_intensity_score,traffic_intensity_score,flow_intensity,traffic_imbalance,speed_score,density_score,syn_flag_count,ack_flag_count,rst_flag_count,psh_flag_count,fin_flag_count,ece_flag_count,cwe_flag_count,tcp_control_total,handshake_signal_score,reset_signal_score,push_activity_score,initial_window_bytes_backward,down_up_ratio,flow_stability,consistency_score,session_size_score,session_packet_score,session_efficiency_score,complexity_score,complexity_score_v2,entropy_score,overall_behavior_score
0,1,Adult,3way,seen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,mixed_behavior,37034,443,1,0,1,0,0,0.143,0.142,0.108,0,12,12,24,540.0,504,1044.0,167.832168,7300.699301,43.500000,485.011577,2291.203222,43.500000,167.832168,1.071287,1.0000,1.071287,0.000000,0.034483,0.482759,0.500000,1.000000,0.965517,0.004842,0.014267,0.014267,0.105400,0.0,0.008872,0.003182,0.018974,0.106505,0.0,0.007420,0.035234,0.0,0.014198,1.100073,2.382923,0.099769,0.995181,120.0,142.0,0.0,0.0,0.0,0,0.0,0.0,0.0,10.0,12.0,0.0,0.0,1.0,0.821839,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.024768,540.230908,50752.79818,8.564294,7250.000000,36.0,8.895862,6.101906,1,1,0,0,0,0,0,2,2,0,0,8190.5,0.0,0.496475,0.820664,6.951772,3.218876,43.500000,0.717196,2.319333,0.577132,0.820649
1,2,Adult,3way,seen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,mixed_behavior,37796,443,1,0,1,0,0,0.195,0.187,0.148,0,12,15,27,540.0,624,1164.0,138.461538,5969.230769,43.111111,85.281584,210.285688,43.111111,138.461538,0.865600,0.8125,0.865600,-0.111111,-0.072165,0.536082,0.555556,0.888889,0.927835,0.006545,0.018651,0.018651,0.133277,0.0,0.011661,0.012397,0.024075,0.135247,0.0,0.025961,0.136674,0.0,0.018530,1.125908,2.565654,0.095646,0.993498,120.0,152.0,0.0,0.0,0.0,0,0.0,0.0,0.0,10.0,13.0,0.0,0.0,1.0,0.772241,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.632330,461.382161,42145.61277,8.392651,5938.775510,84.0,8.694541,5.927521,1,1,0,0,0,0,0,2,2,0,0,8191.5,0.5,0.495410,0.770711,7.060476,3.332205,43.111111,0.789278,2.392480,0.576152,0.791873
2,3,Adult,3way,seen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,mixed_behavior,39998,443,1,0,1,0,0,0.205,0.195,0.158,0,12,15,27,540.0,624,1164.0,131.707317,5678.048780,43.111111,72.616746,174.370864,43.111111,131.707317,0.865600,0.8125,0.865600,-0.111111,-0.072165,0.536082,0.555556,0.888889,0.927835,0.007111,0.020297,0.020297,0.142927,0.0,0.012218,0.013376,0.025678,0.144754,0.0,0.028109,0.145861,0.0,0.020154,1.134857,2.654388,0.099010,0.992939,120.0,152.0,0.0,0.0,0.0,0,0.

## Step 10: Show The Last 5 Rows

This helps confirm the file looks good from top to bottom.


In [10]:
display(df.tail())


,record_id,Category,app_name,split,transport_protocol,ip_version,protocol_family,flow_pattern_guess,traffic_type,traffic_type_confidence,traffic_type_is_guessed,app_behavior_hint,src_port,dst_port,is_tcp,is_udp,is_https,is_dns,is_http,duration_seconds,forward_duration_seconds,backward_duration_seconds,long_flow,orig_packets,resp_packets,total_packets,orig_bytes,resp_bytes,total_bytes,packets_per_second,bytes_per_second,bytes_per_packet,forward_packets_per_second,backward_packets_per_second,throughput_ratio,flow_density,upload_download_ratio,packet_ratio,byte_ratio,directional_packet_ratio,directional_byte_ratio,server_dominance,response_ratio,flow_symmetry,interaction_balance,flow_iat_mean_seconds,flow_iat_std_seconds,inter_arrival_variation,flow_iat_max_seconds,flow_iat_min_seconds,fwd_iat_mean_seconds,bwd_iat_mean_seconds,forward_iat_std_seconds,forward_iat_max_seconds,forward_iat_min_seconds,backward_iat_std_seconds,backward_iat_max_seconds,backward_iat_min_seconds,burst_score,burst_ratio,burst_intensity,time_variability_score,latency_score,forward_header_length,backward_header_length,pkt_len_mean,pkt_len_std,packet_size_variance,pkt_len_min,pkt_len_max,fwd_pkt_len_mean,bwd_pkt_len_mean,subflow_forward_packets,subflow_backward_packets,subflow_forward_bytes,subflow_backward_bytes,packet_variation_score,shape_score,active_mean_seconds,active_std_seconds,active_max_seconds,active_min_seconds,idle_mean_seconds,idle_std_seconds,idle_max_seconds,idle_min_seconds,activity_score,interaction_intensity,packet_intensity_score,byte_intensity_score,traffic_intensity_score,flow_intensity,traffic_imbalance,speed_score,density_score,syn_flag_count,ack_flag_count,rst_flag_count,psh_flag_count,fin_flag_count,ece_flag_count,cwe_flag_count,tcp_control_total,handshake_signal_score,reset_signal_score,push_activity_score,initial_window_bytes_backward,down_up_ratio,flow_stability,consistency_score,session_size_score,session_packet_score,session_efficiency_score,complexity_score,complexity_score_v2,entropy_score,overall_behavior_score
62893,62894,Social-Media,zeli,unseen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,server_response_heavy,49860,443,1,0,1,0,0,24.026,23.987,24.019,1,549,522,1071,22020.0,2226270,2248290.0,44.576709,93577.374510,2099.243697,24.115411,48.978877,2099.243677,44.576709,0.009891,1.051625,0.009891,0.025210,-0.980412,0.990206,0.487395,0.974790,0.019588,0.017837,0.296266,0.296266,18.385625,0.0,0.022006,0.015242,0.398465,18.385625,0.0,0.072424,2.529387,0.0,0.291074,19.045903,12.975126,0.012331,0.982476,5490.0,5222.0,1036.356203,1689.132498,1689.132498,0,9960.0,0.0,2132.84913,547.0,520.0,0.0,2205366.0,1.629269,0.458237,0.0000,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.0,0.000000,15.265951,311.024240,1.368633e+06,9.936202,93573.479840,2204250.0,11.446555,0.584438,1,1,0,0,0,0,0,2,2,0,0,8191.5,0.0,0.342425,0.445601,14.625681,6.977281,2099.243697,1.367612,3.123799,0.042902,0.560364
62894,62895,Social-Media,zeli,unseen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,server_response_heavy,49864,443,1,0,1,0,0,24.017,23.978,24.011,1,216,69,285,8700.0,59904,68604.0,11.866594,2856.476662,240.715789,17.083290,39.259354,240.715799,11.866594,0.145247,3.100000,0.145247,0.515789,-0.746370,0.873185,0.242105,0.484211,0.253630,0.050029,0.571739,0.571739,18.396868,0.0,0.056550,0.054172,0.639621,18.396868,0.0,0.200903,2.497874,0.0,0.544498,18.472697,6.461340,0.023806,0.952355,2160.0,692.0,102.733813,258.273596,258.273596,0,2856.0,0.0,446.25000,214.0,67.0,0.0,57120.0,2.499413,0.341201,0.0000,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.0,0.000000,10.512329,67.117358,3.181007e+04,7.294401,2856.357732,51204.0,7.957694,0.445308,1,1,0,0,0,0,0,2,2,0,0,8191.5,0.0,0.247285,0.328375,11.136121,5.655992,240.715789,1.668852,3.520179,0.294191,0.504573
62895,62896,Social-Media,zeli,unseen,tcp,4,tcp_flow,secure_web_flow,Web,1,1,server_response_heavy,49868,443,1,0,1,0,0,24.006,23.965,24.001,1,69,40,109,2820.0,34468,37288.0,4.540532,1553.278347,342.091743,13.643918,37.531007,342.091708,4.5

## Step 11: Show All Column Names

This table lists every header in the CSV.


In [11]:
column_table = pd.DataFrame(
    {
        "column_number": range(1, len(df.columns) + 1),
        "column_name": df.columns,
    }
)

display(column_table)


,column_number,column_name
0,1,record_id
1,2,Category
2,3,app_name
3,4,split
4,5,transport_protocol
...,...,...
112,113,session_efficiency_score
113,114,complexity_score
114,115,complexity_score_v2
115,116,entropy_score


## Step 12: Show A Few Important Columns Together

These columns help us understand the identity and meaning of each row.


In [12]:
important_columns = [
    "record_id",
    "Category",
    "app_name",
    "split",
    "transport_protocol",
    "traffic_type",
    "flow_pattern_guess",
    "overall_behavior_score",
]

display(df[important_columns].head(10))


,record_id,Category,app_name,split,transport_protocol,traffic_type,flow_pattern_guess,overall_behavior_score
0,1,Adult,3way,seen,tcp,Web,secure_web_flow,0.820649
1,2,Adult,3way,seen,tcp,Web,secure_web_flow,0.791873
2,3,Adult,3way,seen,tcp,Web,secure_web_flow,0.790880
3,4,Adult,3way,seen,tcp,Web,secure_web_flow,0.797436
4,5,Adult,3way,seen,tcp,Web,secure_web_flow,0.812842
5,6,Adult,3way,seen,tcp,Web,secure_web_flow,0.820661
6,7,Adult,3way,seen,tcp,Web,secure_web_flow,0.793663
7,8,Adult,3way,seen,tcp,Web,secure_web_flow,0.795486
8,9,Adult,3way,seen,tcp,Web,download_heavy_flow,0.415071
9,10,Adult,3way,seen,tcp,Web,secure_web_flow,0.719472


## Step 13: Show Data Types Summary

This tells us how many columns are integer, float, or text.


In [13]:
dtype_summary = (
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="column_count")
)

display(dtype_summary)


,dtype,column_count
0,float64,81
1,int64,28
2,object,8


## Step 14: Show Full `df.info()` Output

This is a standard pandas overview of the whole dataframe.


In [14]:
info_buffer = StringIO()
df.info(buf=info_buffer)
print(info_buffer.getvalue())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62898 entries, 0 to 62897
Columns: 117 entries, record_id to overall_behavior_score
dtypes: float64(81), int64(28), object(8)
memory usage: 56.1+ MB



## Step 15: Show The Category Labels

These are the class names your model will learn later.


In [15]:
category_values = sorted(df[target_col].dropna().unique().tolist())
print("Category labels found in the dataset:")
print(category_values)


Category labels found in the dataset:
['Adult', 'Game', 'Payment', 'Social-Media']


## Step 16: Count Rows In Each Category

This tells us how many rows belong to each main class.


In [16]:
category_counts = (
    df[target_col]
    .value_counts()
    .rename_axis(target_col)
    .reset_index(name="row_count")
)

display(category_counts)


,Category,row_count
0,Game,22019
1,Social-Media,18520
2,Payment,13347
3,Adult,9012


## Step 17: Show Category Percentages

Percentages make class balance easier to understand.


In [17]:
category_percentages = (
    df[target_col]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis(target_col)
    .reset_index(name="percentage")
)

display(category_percentages)


,Category,percentage
0,Game,35.01
1,Social-Media,29.44
2,Payment,21.22
3,Adult,14.33


## Step 18: Install `scikit-learn` Only If Needed

We are going to train several machine-learning models in this notebook.

Run the next cell only if the import step gives an error.


In [18]:
# Run the commented line only if Step 19 gives an import error in your own environment.
# %pip install scikit-learn
print("Optional install cell skipped because scikit-learn is already available in this environment.")


Optional install cell skipped because scikit-learn is already available in this environment.


## Step 19: Import The Modeling And Timing Tools

We now import the libraries for:
- preprocessing
- train/test splitting
- model tuning
- evaluation metrics
- timing

This notebook uses one combined dataset and a stratified 80:20 split.


In [19]:
import time
import warnings

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("Modeling libraries imported successfully.")


Modeling libraries imported successfully.


## Step 20: Explain The New Research Design

This notebook is different from the earlier seen/unseen workflow.

Our new research design is:
1. use one combined dataset
2. remove exact duplicates
3. audit app balance and category balance
4. drop `record_id`, `app_name`, and `split` from model input features
5. balance categories before the split
6. create one stratified 80:20 train/test split
7. tune 5 lightweight single models
8. compare 5 hard-voting hybrid models
9. choose one best lightweight single model and one best hybrid model for later C conversion


In [20]:
random_seed = 42

redesign_summary = pd.DataFrame(
    [
        {"decision": "Main target", "value": target_col},
        {"decision": "Source CSV changed?", "value": "No"},
        {"decision": "Use raw seen/unseen split for modeling?", "value": "No"},
        {"decision": "Main experiment split", "value": "Stratified 80:20 train/test"},
        {"decision": "Balance timing", "value": "Before the train/test split"},
        {"decision": "Balance method", "value": "Downsample each category to the smallest category size"},
        {"decision": "Main ranking metric", "value": "Test macro F1"},
    ]
)

display(redesign_summary)


,decision,value
0,Main target,Category
1,Source CSV changed?,No
2,Use raw seen/unseen split for modeling?,No
3,Main experiment split,Stratified 80:20 train/test
4,Balance timing,Before the train/test split
5,Balance method,Downsample each category to the smallest category size
6,Main ranking metric,Test macro F1


## Step 21: Build A Notebook-Only Working Copy

We do not modify the original CSV file on disk.

Instead, we create a working DataFrame inside the notebook.


In [21]:
working_df = df.copy()

working_copy_summary = pd.DataFrame(
    [
        {"item": "Working copy rows", "value": int(working_df.shape[0])},
        {"item": "Working copy columns", "value": int(working_df.shape[1])},
        {"item": "Target column", "value": target_col},
        {"item": "Source split column still present?", "value": "Yes"},
    ]
)

display(working_copy_summary)


,item,value
0,Working copy rows,62898
1,Working copy columns,117
2,Target column,Category
3,Source split column still present?,Yes


## Step 22: Remove Exact Duplicate Rows

Why this matters:
- duplicate rows can make a model look better than it really is
- duplicates can also distort the class balance
- we want the train/test split to use unique rows only


In [22]:
duplicate_row_count = int(working_df.duplicated().sum())
dedup_df = working_df.drop_duplicates().reset_index(drop=True)

duplicate_summary = pd.DataFrame(
    [
        {"item": "Rows before deduplication", "value": int(working_df.shape[0])},
        {"item": "Exact duplicate rows found", "value": duplicate_row_count},
        {"item": "Rows after deduplication", "value": int(dedup_df.shape[0])},
    ]
)

display(duplicate_summary)


,item,value
0,Rows before deduplication,62898
1,Exact duplicate rows found,0
2,Rows after deduplication,62898


## Step 23: Audit App-Level Row Counts Before Dropping `app_name`

We will not use `app_name` as a model feature.

But before dropping it, we should still inspect it because:
- it tells us whether some apps dominate the dataset
- it helps us explain the raw data quality
- it gives research context before we switch to pure category prediction


In [23]:
app_counts_before_drop = (
    dedup_df["app_name"]
    .value_counts()
    .rename_axis("app_name")
    .reset_index(name="row_count")
)

app_balance_summary = pd.DataFrame(
    [
        {"item": "Unique apps", "value": int(app_counts_before_drop.shape[0])},
        {"item": "Smallest app row count", "value": int(app_counts_before_drop["row_count"].min())},
        {"item": "Median app row count", "value": float(app_counts_before_drop["row_count"].median())},
        {"item": "Largest app row count", "value": int(app_counts_before_drop["row_count"].max())},
    ]
)

print("App balance summary:")
display(app_balance_summary)
print("Top 20 apps by row count:")
display(app_counts_before_drop.head(20))
print("Bottom 20 apps by row count:")
display(app_counts_before_drop.tail(20))


App balance summary:


,item,value
0,Unique apps,200.0
1,Smallest app row count,1.0
2,Median app row count,141.5
3,Largest app row count,6373.0


Top 20 apps by row count:


,app_name,row_count
0,alipay,6373
1,clash_of_clans,3957
2,pubg_mobile,3512
3,wechat,3427
4,meituan,2084
5,top_eleven,1352
6,clash_royale,1324
7,chamet,1074
8,palmchat,941
9,8_ball_pool,910


Bottom 20 apps by row count:


,app_name,row_count
180,bikroy,28
181,meesho,25
182,wakie,25
183,raid,21
184,brac_bank_astha,19
185,pinterest,18
186,matchwatch,17
187,brawlhalla,16
188,clubhouse,14
189,meet,12


## Step 24: Explain Which Columns Will Be Dropped Before Modeling

We now separate:
- columns useful only for tracking or reporting
- the target label
- columns that will actually be used for model training

In this notebook:
- `record_id` is only an identifier
- `app_name` would leak direct app identity
- `split` belongs to the source CSV design, but not to this new 80:20 experiment


In [24]:
drop_from_features = ["record_id", "app_name", "split"]
modeling_df = dedup_df.drop(columns=drop_from_features).copy()

column_role_table = pd.DataFrame(
    [
        {"column_name": "record_id", "role": "drop from X", "reason": "Identifier only"},
        {"column_name": "app_name", "role": "drop from X", "reason": "Direct app identity should not be used as a feature"},
        {"column_name": "split", "role": "drop from X", "reason": "This notebook does not use the original seen/unseen design"},
        {"column_name": target_col, "role": "keep as y only", "reason": "This is the category label to predict"},
    ]
)

display(column_role_table)

modeling_copy_summary = pd.DataFrame(
    [
        {"item": "Rows in modeling_df", "value": int(modeling_df.shape[0])},
        {"item": "Columns in modeling_df", "value": int(modeling_df.shape[1])},
        {"item": "Feature columns after dropping non-model inputs", "value": int(modeling_df.shape[1] - 1)},
    ]
)

display(modeling_copy_summary)


,column_name,role,reason
0,record_id,drop from X,Identifier only
1,app_name,drop from X,Direct app identity should not be used as a feature
2,split,drop from X,This notebook does not use the original seen/unseen design
3,Category,keep as y only,This is the category label to predict


,item,value
0,Rows in modeling_df,62898
1,Columns in modeling_df,114
2,Feature columns after dropping non-model inputs,113


## Step 25: Show Category Balance Before Balancing

The main target is `Category`, so class balance matters directly.

We inspect the raw category balance now, before any balancing step.


In [25]:
category_balance_before = (
    modeling_df[target_col]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="row_count")
)
category_balance_before["row_pct"] = (
    category_balance_before["row_count"] / category_balance_before["row_count"].sum() * 100
).round(2)
category_balance_before["vs_smallest_class"] = (
    category_balance_before["row_count"] / category_balance_before["row_count"].min()
).round(2)

display(category_balance_before)


,Category,row_count,row_pct,vs_smallest_class
0,Game,22019,35.01,2.44
1,Social-Media,18520,29.44,2.06
2,Payment,13347,21.22,1.48
3,Adult,9012,14.33,1.00


## Step 26: Balance Categories Before The Split

Why balancing is done before the split in this notebook:
- this is the workflow requested for this research notebook
- it gives a clean category-balanced dataset
- it lets us compare models under the same balanced class distribution

Important note:
- this choice can make the final test set easier than the raw dataset
- so the results should be read as a controlled research comparison, not as a strict real-world deployment benchmark


In [26]:
smallest_category_size = int(category_balance_before["row_count"].min())

balanced_df = (
    modeling_df.groupby(target_col, group_keys=False)
    .apply(lambda frame: frame.sample(n=smallest_category_size, random_state=random_seed))
    .reset_index(drop=True)
)

balancing_summary = pd.DataFrame(
    [
        {"item": "Smallest category size used for balancing", "value": smallest_category_size},
        {"item": "Rows before balancing", "value": int(modeling_df.shape[0])},
        {"item": "Rows after balancing", "value": int(balanced_df.shape[0])},
    ]
)

display(balancing_summary)


C:\Users\Sohan\AppData\Local\Temp\ipykernel_12988\4095748480.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda frame: frame.sample(n=smallest_category_size, random_state=random_seed))


,item,value
0,Smallest category size used for balancing,9012
1,Rows before balancing,62898
2,Rows after balancing,36048


## Step 27: Show Category Balance After Balancing

This table should now show equal row counts for all categories.


In [27]:
category_balance_after = (
    balanced_df[target_col]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="row_count")
)
category_balance_after["row_pct"] = (
    category_balance_after["row_count"] / category_balance_after["row_count"].sum() * 100
).round(2)

display(category_balance_after)


,Category,row_count,row_pct
0,Adult,9012,25.0
1,Game,9012,25.0
2,Payment,9012,25.0
3,Social-Media,9012,25.0


## Step 28: Build The Final Feature Matrix And Target Vector

We now create:
- `X_all` for features
- `y_all` for labels

We also split the feature columns into:
- numeric columns
- categorical columns


In [28]:
X_all = balanced_df.drop(columns=[target_col]).copy()
y_all = balanced_df[target_col].copy()

categorical_columns = X_all.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
numeric_columns = [column for column in X_all.columns if column not in categorical_columns]

feature_build_summary = pd.DataFrame(
    [
        {"item": "Balanced rows", "value": int(X_all.shape[0])},
        {"item": "Feature columns", "value": int(X_all.shape[1])},
        {"item": "Numeric feature columns", "value": int(len(numeric_columns))},
        {"item": "Categorical feature columns", "value": int(len(categorical_columns))},
    ]
)

display(feature_build_summary)
print("First 15 categorical columns:")
display(pd.DataFrame({"categorical_column": categorical_columns[:15]}))
print("First 15 numeric columns:")
display(pd.DataFrame({"numeric_column": numeric_columns[:15]}))


,item,value
0,Balanced rows,36048
1,Feature columns,113
2,Numeric feature columns,108
3,Categorical feature columns,5


First 15 categorical columns:


,categorical_column
0,transport_protocol
1,protocol_family
2,flow_pattern_guess
3,traffic_type
4,app_behavior_hint


First 15 numeric columns:


,numeric_column
0,ip_version
1,traffic_type_confidence
2,traffic_type_is_guessed
3,src_port
4,dst_port
5,is_tcp
6,is_udp
7,is_https
8,is_dns
9,is_http


## Step 29: Create The Stratified 80:20 Train/Test Split

We now split the balanced category dataset into:
- 80% training data
- 20% testing data

We use stratification so that each category stays equally represented in both parts.


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=random_seed,
)

split_summary = pd.DataFrame(
    [
        {"subset": "Train", "rows": int(X_train.shape[0]), "row_pct": round(X_train.shape[0] / X_all.shape[0] * 100, 2)},
        {"subset": "Test", "rows": int(X_test.shape[0]), "row_pct": round(X_test.shape[0] / X_all.shape[0] * 100, 2)},
    ]
)

train_test_balance = pd.DataFrame(
    {
        "train_count": y_train.value_counts().sort_index(),
        "test_count": y_test.value_counts().sort_index(),
        "train_pct": y_train.value_counts(normalize=True).sort_index().mul(100).round(2),
        "test_pct": y_test.value_counts(normalize=True).sort_index().mul(100).round(2),
    }
).reset_index(names="Category")

display(split_summary)
display(train_test_balance)


,subset,rows,row_pct
0,Train,28838,80.0
1,Test,7210,20.0


,Category,train_count,test_count,train_pct,test_pct
0,Adult,7209,1803,25.0,25.01
1,Game,7210,1802,25.0,24.99
2,Payment,7209,1803,25.0,25.01
3,Social-Media,7210,1802,25.0,24.99


## Step 30: Explain Why We Use Macro F1 And Model-Specific Tuning

Accuracy alone is not enough, even after balancing.

We will report:
- accuracy
- balanced accuracy
- macro precision
- macro recall
- macro F1
- weighted F1
- MCC
- fit time
- prediction time

Also, epoch and learning-rate tuning only make sense for SGD-style optimization.
Other models have their own equivalent knobs:
- Logistic Regression: `C` and `max_iter`
- Linear SVC: `C` and `max_iter`
- Decision Tree: `max_depth` and `min_samples_leaf`
- Random Forest: number of trees, depth, and leaf size


In [30]:
evaluation_design_table = pd.DataFrame(
    [
        {"topic": "Main ranking metric", "choice": "Test macro F1"},
        {"topic": "Secondary ranking metric", "choice": "Test accuracy"},
        {"topic": "Tertiary ranking metric", "choice": "Lower prediction time"},
        {"topic": "Cross-validation", "choice": "3-fold StratifiedKFold on the training split only"},
        {"topic": "Why SGD has learning rate and max_iter", "choice": "It learns iteratively and those are natural optimization controls"},
        {"topic": "Why tree models use depth and leaf controls", "choice": "Tree capacity is controlled by structure, not by epochs"},
    ]
)

display(evaluation_design_table)


,topic,choice
0,Main ranking metric,Test macro F1
1,Secondary ranking metric,Test accuracy
2,Tertiary ranking metric,Lower prediction time
3,Cross-validation,3-fold StratifiedKFold on the training split only
4,Why SGD has learning rate and max_iter,It learns iteratively and those are natural optimization controls
5,Why tree models use depth and leaf controls,"Tree capacity is controlled by structure, not by epochs"


## Step 31: Define Preprocessing Helpers And Evaluation Functions

We now build reusable tools for:
- preprocessing
- timing
- model scoring
- one-vs-rest positive and negative metrics


In [31]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_seed)
class_labels = sorted(y_all.unique().tolist())


def build_linear_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


def build_tree_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                numeric_columns,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


def compute_metric_row(model_name, model_family, resource_class, c_export_friendliness, y_true, y_pred, fit_time_seconds, prediction_time_seconds):
    return {
        "model_name": model_name,
        "model_family": model_family,
        "resource_class": resource_class,
        "c_export_friendliness": c_export_friendliness,
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        "macro_precision": round(float(precision_score(y_true, y_pred, average="macro", zero_division=0)), 4),
        "macro_recall": round(float(recall_score(y_true, y_pred, average="macro", zero_division=0)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4),
        "weighted_f1": round(float(f1_score(y_true, y_pred, average="weighted", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y_true, y_pred)), 4),
        "fit_time_seconds": round(float(fit_time_seconds), 4),
        "prediction_time_seconds": round(float(prediction_time_seconds), 6),
    }


def build_one_vs_rest_table(model_name, y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    total = int(cm.sum())
    rows = []

    for index, label in enumerate(labels):
        tp = int(cm[index, index])
        fn = int(cm[index, :].sum() - tp)
        fp = int(cm[:, index].sum() - tp)
        tn = int(total - tp - fn - fp)

        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        false_positive_rate = fp / (fp + tn) if (fp + tn) else np.nan
        false_negative_rate = fn / (fn + tp) if (fn + tp) else np.nan
        negative_predictive_value = tn / (tn + fn) if (tn + fn) else np.nan

        rows.append(
            {
                "model_name": model_name,
                "class_label": label,
                "tp": tp,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "specificity": round(float(specificity), 4),
                "false_positive_rate": round(float(false_positive_rate), 4),
                "false_negative_rate": round(float(false_negative_rate), 4),
                "negative_predictive_value": round(float(negative_predictive_value), 4),
            }
        )

    return pd.DataFrame(rows)


## Step 32: Define The Five Lightweight Single-Model Search Spaces

We now prepare the five single-model candidates:
1. Logistic Regression
2. SGD Log-Loss Classifier
3. Linear SVC
4. Decision Tree
5. Compact Random Forest


In [32]:
single_model_specs = {
    "LogisticRegression": {
        "model_family": "Linear",
        "resource_class": "Very light",
        "c_export_friendliness": "Very high",
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", build_linear_preprocessor()),
                ("model", LogisticRegression(solver="saga", random_state=random_seed)),
            ]
        ),
        "param_grid": {
            "model__C": [0.25, 0.5, 1.0, 2.0],
            "model__max_iter": [1000, 2000],
        },
    },
    "SGDLogLoss": {
        "model_family": "Linear",
        "resource_class": "Very light",
        "c_export_friendliness": "Very high",
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", build_linear_preprocessor()),
                ("model", SGDClassifier(loss="log_loss", random_state=random_seed)),
            ]
        ),
        "param_grid": {
            "model__learning_rate": ["adaptive"],
            "model__eta0": [0.001, 0.005, 0.01, 0.05],
            "model__max_iter": [500, 1000, 2000],
        },
    },
    "LinearSVC": {
        "model_family": "Linear",
        "resource_class": "Very light",
        "c_export_friendliness": "Very high",
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", build_linear_preprocessor()),
                ("model", LinearSVC(random_state=random_seed)),
            ]
        ),
        "param_grid": {
            "model__C": [0.25, 0.5, 1.0, 2.0],
            "model__max_iter": [2000, 5000],
        },
    },
    "DecisionTree": {
        "model_family": "Tree",
        "resource_class": "Light",
        "c_export_friendliness": "Very high",
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", build_tree_preprocessor()),
                ("model", DecisionTreeClassifier(random_state=random_seed)),
            ]
        ),
        "param_grid": {
            "model__max_depth": [6, 10, 14],
            "model__min_samples_leaf": [1, 3, 5],
        },
    },
    "CompactRF": {
        "model_family": "Tree",
        "resource_class": "Medium",
        "c_export_friendliness": "Medium",
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", build_tree_preprocessor()),
                ("model", RandomForestClassifier(random_state=random_seed, n_jobs=-1)),
            ]
        ),
        "param_grid": {
            "model__n_estimators": [10, 20],
            "model__max_depth": [10, 14],
            "model__min_samples_leaf": [1, 3, 5],
        },
    },
}

single_model_catalog = pd.DataFrame(
    [
        {
            "model_name": model_name,
            "model_family": spec["model_family"],
            "resource_class": spec["resource_class"],
            "c_export_friendliness": spec["c_export_friendliness"],
            "search_grid_size": int(np.prod([len(values) for values in spec["param_grid"].values()])),
        }
        for model_name, spec in single_model_specs.items()
    ]
)

display(single_model_catalog)


,model_name,model_family,resource_class,c_export_friendliness,search_grid_size
0,LogisticRegression,Linear,Very light,Very high,8
1,SGDLogLoss,Linear,Very light,Very high,12
2,LinearSVC,Linear,Very light,Very high,8
3,DecisionTree,Tree,Light,Very high,9
4,CompactRF,Tree,Medium,Medium,12


## Step 33: Create Empty Tracking Tables For The Single Models

Before we train the models, we create empty notebook variables to store:
- test metrics
- best hyperparameters
- fitted estimators
- predictions
- fit and prediction timing


In [33]:
single_model_rows = []
single_model_best_params_rows = []
best_single_estimators = {}
single_model_predictions = {}
single_model_fit_times = {}
single_model_prediction_times = {}


## Step 34: Train And Evaluate Logistic Regression

This is a strong lightweight baseline. It is linear, compact, and often a good first candidate for later C conversion.

This cell does four things for `LogisticRegression`:
- runs grid search only on the training split
- chooses the best hyperparameters using macro F1
- predicts on the 20% test split
- shows both the winning settings and the final test metrics


In [34]:
current_model_name = "LogisticRegression"
current_spec = single_model_specs[current_model_name]

current_grid_search = GridSearchCV(
    estimator=current_spec["pipeline"],
    param_grid=current_spec["param_grid"],
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

fit_start = time.perf_counter()
current_grid_search.fit(X_train, y_train)
fit_time = time.perf_counter() - fit_start

predict_start = time.perf_counter()
current_y_pred = current_grid_search.best_estimator_.predict(X_test)
prediction_time = time.perf_counter() - predict_start

current_result_row = {
    **compute_metric_row(
        model_name=current_model_name,
        model_family=current_spec["model_family"],
        resource_class=current_spec["resource_class"],
        c_export_friendliness=current_spec["c_export_friendliness"],
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
}

current_best_params_row = {
    "model_name": current_model_name,
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
    "best_params": str(current_grid_search.best_params_),
}

single_model_rows.append(current_result_row)
single_model_best_params_rows.append(current_best_params_row)
best_single_estimators[current_model_name] = current_grid_search.best_estimator_
single_model_predictions[current_model_name] = current_y_pred
single_model_fit_times[current_model_name] = fit_time
single_model_prediction_times[current_model_name] = prediction_time

current_best_params_table = pd.DataFrame([current_best_params_row])
current_result_table = pd.DataFrame([current_result_row])

print(f"Finished {current_model_name}.")
display(current_best_params_table)
display(current_result_table)


Finished LogisticRegression.


,model_name,best_cv_macro_f1,best_params
0,LogisticRegression,0.6636,"{'model__C': 2.0, 'model__max_iter': 2000}"


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,LogisticRegression,Linear,Very light,Very high,0.6635,0.6635,0.6627,0.6635,0.661,0.661,0.5525,502.5065,0.025058,0.6636


## Step 35: Train And Evaluate SGD Log-Loss Classifier

This model learns in small iterative updates, so this is the place where learning rate and max_iter behave most like the beginner idea of epochs and step size.

This cell does four things for `SGDLogLoss`:
- runs grid search only on the training split
- chooses the best hyperparameters using macro F1
- predicts on the 20% test split
- shows both the winning settings and the final test metrics


In [35]:
current_model_name = "SGDLogLoss"
current_spec = single_model_specs[current_model_name]

current_grid_search = GridSearchCV(
    estimator=current_spec["pipeline"],
    param_grid=current_spec["param_grid"],
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

fit_start = time.perf_counter()
current_grid_search.fit(X_train, y_train)
fit_time = time.perf_counter() - fit_start

predict_start = time.perf_counter()
current_y_pred = current_grid_search.best_estimator_.predict(X_test)
prediction_time = time.perf_counter() - predict_start

current_result_row = {
    **compute_metric_row(
        model_name=current_model_name,
        model_family=current_spec["model_family"],
        resource_class=current_spec["resource_class"],
        c_export_friendliness=current_spec["c_export_friendliness"],
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
}

current_best_params_row = {
    "model_name": current_model_name,
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
    "best_params": str(current_grid_search.best_params_),
}

single_model_rows.append(current_result_row)
single_model_best_params_rows.append(current_best_params_row)
best_single_estimators[current_model_name] = current_grid_search.best_estimator_
single_model_predictions[current_model_name] = current_y_pred
single_model_fit_times[current_model_name] = fit_time
single_model_prediction_times[current_model_name] = prediction_time

current_best_params_table = pd.DataFrame([current_best_params_row])
current_result_table = pd.DataFrame([current_result_row])

print(f"Finished {current_model_name}.")
display(current_best_params_table)
display(current_result_table)


Finished SGDLogLoss.


,model_name,best_cv_macro_f1,best_params
0,SGDLogLoss,0.6671,"{'model__eta0': 0.05, 'model__learning_rate': 'adaptive', 'model__max_iter':..."


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,SGDLogLoss,Linear,Very light,Very high,0.6666,0.6666,0.6666,0.6666,0.6644,0.6644,0.5566,19.3367,0.024058,0.6671


## Step 36: Train And Evaluate Linear SVC

This model is also lightweight, but it optimizes a margin-based objective instead of log-loss.

This cell does four things for `LinearSVC`:
- runs grid search only on the training split
- chooses the best hyperparameters using macro F1
- predicts on the 20% test split
- shows both the winning settings and the final test metrics


In [36]:
current_model_name = "LinearSVC"
current_spec = single_model_specs[current_model_name]

current_grid_search = GridSearchCV(
    estimator=current_spec["pipeline"],
    param_grid=current_spec["param_grid"],
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

fit_start = time.perf_counter()
current_grid_search.fit(X_train, y_train)
fit_time = time.perf_counter() - fit_start

predict_start = time.perf_counter()
current_y_pred = current_grid_search.best_estimator_.predict(X_test)
prediction_time = time.perf_counter() - predict_start

current_result_row = {
    **compute_metric_row(
        model_name=current_model_name,
        model_family=current_spec["model_family"],
        resource_class=current_spec["resource_class"],
        c_export_friendliness=current_spec["c_export_friendliness"],
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
}

current_best_params_row = {
    "model_name": current_model_name,
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
    "best_params": str(current_grid_search.best_params_),
}

single_model_rows.append(current_result_row)
single_model_best_params_rows.append(current_best_params_row)
best_single_estimators[current_model_name] = current_grid_search.best_estimator_
single_model_predictions[current_model_name] = current_y_pred
single_model_fit_times[current_model_name] = fit_time
single_model_prediction_times[current_model_name] = prediction_time

current_best_params_table = pd.DataFrame([current_best_params_row])
current_result_table = pd.DataFrame([current_result_row])

print(f"Finished {current_model_name}.")
display(current_best_params_table)
display(current_result_table)


c:\Users\Sohan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Finished LinearSVC.


,model_name,best_cv_macro_f1,best_params
0,LinearSVC,0.6713,"{'model__C': 2.0, 'model__max_iter': 2000}"


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,LinearSVC,Linear,Very light,Very high,0.6718,0.6718,0.6722,0.6718,0.6703,0.6703,0.5634,699.3149,0.026041,0.6713


## Step 37: Train And Evaluate Decision Tree

This model is easy to explain because it makes rule-like splits, which can also make manual conversion into C more approachable.

This cell does four things for `DecisionTree`:
- runs grid search only on the training split
- chooses the best hyperparameters using macro F1
- predicts on the 20% test split
- shows both the winning settings and the final test metrics


In [37]:
current_model_name = "DecisionTree"
current_spec = single_model_specs[current_model_name]

current_grid_search = GridSearchCV(
    estimator=current_spec["pipeline"],
    param_grid=current_spec["param_grid"],
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

fit_start = time.perf_counter()
current_grid_search.fit(X_train, y_train)
fit_time = time.perf_counter() - fit_start

predict_start = time.perf_counter()
current_y_pred = current_grid_search.best_estimator_.predict(X_test)
prediction_time = time.perf_counter() - predict_start

current_result_row = {
    **compute_metric_row(
        model_name=current_model_name,
        model_family=current_spec["model_family"],
        resource_class=current_spec["resource_class"],
        c_export_friendliness=current_spec["c_export_friendliness"],
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
}

current_best_params_row = {
    "model_name": current_model_name,
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
    "best_params": str(current_grid_search.best_params_),
}

single_model_rows.append(current_result_row)
single_model_best_params_rows.append(current_best_params_row)
best_single_estimators[current_model_name] = current_grid_search.best_estimator_
single_model_predictions[current_model_name] = current_y_pred
single_model_fit_times[current_model_name] = fit_time
single_model_prediction_times[current_model_name] = prediction_time

current_best_params_table = pd.DataFrame([current_best_params_row])
current_result_table = pd.DataFrame([current_result_row])

print(f"Finished {current_model_name}.")
display(current_best_params_table)
display(current_result_table)


Finished DecisionTree.


,model_name,best_cv_macro_f1,best_params
0,DecisionTree,0.8294,"{'model__max_depth': 14, 'model__min_samples_leaf': 1}"


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,DecisionTree,Tree,Light,Very high,0.8466,0.8466,0.8467,0.8466,0.8461,0.8461,0.7958,8.5457,0.021451,0.8294


## Step 38: Train And Evaluate Compact Random Forest

This model is usually stronger than one single tree, but it is heavier because it combines multiple trees.

This cell does four things for `CompactRF`:
- runs grid search only on the training split
- chooses the best hyperparameters using macro F1
- predicts on the 20% test split
- shows both the winning settings and the final test metrics


In [38]:
current_model_name = "CompactRF"
current_spec = single_model_specs[current_model_name]

current_grid_search = GridSearchCV(
    estimator=current_spec["pipeline"],
    param_grid=current_spec["param_grid"],
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)

fit_start = time.perf_counter()
current_grid_search.fit(X_train, y_train)
fit_time = time.perf_counter() - fit_start

predict_start = time.perf_counter()
current_y_pred = current_grid_search.best_estimator_.predict(X_test)
prediction_time = time.perf_counter() - predict_start

current_result_row = {
    **compute_metric_row(
        model_name=current_model_name,
        model_family=current_spec["model_family"],
        resource_class=current_spec["resource_class"],
        c_export_friendliness=current_spec["c_export_friendliness"],
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
}

current_best_params_row = {
    "model_name": current_model_name,
    "best_cv_macro_f1": round(float(current_grid_search.best_score_), 4),
    "best_params": str(current_grid_search.best_params_),
}

single_model_rows.append(current_result_row)
single_model_best_params_rows.append(current_best_params_row)
best_single_estimators[current_model_name] = current_grid_search.best_estimator_
single_model_predictions[current_model_name] = current_y_pred
single_model_fit_times[current_model_name] = fit_time
single_model_prediction_times[current_model_name] = prediction_time

current_best_params_table = pd.DataFrame([current_best_params_row])
current_result_table = pd.DataFrame([current_result_row])

print(f"Finished {current_model_name}.")
display(current_best_params_table)
display(current_result_table)


Finished CompactRF.


,model_name,best_cv_macro_f1,best_params
0,CompactRF,0.8552,"{'model__max_depth': 14, 'model__min_samples_leaf': 1, 'model__n_estimators'..."


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,CompactRF,Tree,Medium,Medium,0.8663,0.8663,0.8678,0.8663,0.8659,0.8659,0.8225,11.6511,0.055774,0.8552


## Step 39: Build The Single-Model Summary Tables

Now that all 5 single models have been trained one by one, we combine their rows into two clean tables:
- a best-parameter table
- a single-model metric table


In [39]:
single_model_results = pd.DataFrame(single_model_rows)
single_model_best_params = pd.DataFrame(single_model_best_params_rows)

display(single_model_best_params)
display(single_model_results)


,model_name,best_cv_macro_f1,best_params
0,LogisticRegression,0.6636,"{'model__C': 2.0, 'model__max_iter': 2000}"
1,SGDLogLoss,0.6671,"{'model__eta0': 0.05, 'model__learning_rate': 'adaptive', 'model__max_iter':..."
2,LinearSVC,0.6713,"{'model__C': 2.0, 'model__max_iter': 2000}"
3,DecisionTree,0.8294,"{'model__max_depth': 14, 'model__min_samples_leaf': 1}"
4,CompactRF,0.8552,"{'model__max_depth': 14, 'model__min_samples_leaf': 1, 'model__n_estimators'..."


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,LogisticRegression,Linear,Very light,Very high,0.6635,0.6635,0.6627,0.6635,0.6610,0.6610,0.5525,502.5065,0.025058,0.6636
1,SGDLogLoss,Linear,Very light,Very high,0.6666,0.6666,0.6666,0.6666,0.6644,0.6644,0.5566,19.3367,0.024058,0.6671
2,LinearSVC,Linear,Very light,Very high,0.6718,0.6718,0.6722,0.6718,0.6703,0.6703,0.5634,699.3149,0.026041,0.6713
3,DecisionTree,Tree,Light,Very high,0.8466,0.8466,0.8467,0.8466,0.8461,0.8461,0.7958,8.5457,0.021451,0.8294
4,CompactRF,Tree,Medium,Medium,0.8663,0.8663,0.8678,0.8663,0.8659,0.8659,0.8225,11.6511,0.055774,0.8552


## Step 40: Rank The Five Single Models And Choose The Best Lightweight One

We rank the single models by:
1. test macro F1
2. test accuracy
3. lower prediction time


In [40]:
single_model_results_ranked = single_model_results.sort_values(
    ["macro_f1", "accuracy", "prediction_time_seconds"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(single_model_results_ranked)

lightweight_single_candidates = single_model_results_ranked[
    single_model_results_ranked["c_export_friendliness"] == "Very high"
].copy()

best_lightweight_single_row = lightweight_single_candidates.sort_values(
    ["macro_f1", "accuracy", "prediction_time_seconds"],
    ascending=[False, False, True],
).iloc[0]

single_model_choice_table = pd.DataFrame(
    [
        {
            "goal": "Best lightweight single model for later C conversion",
            "model_name": best_lightweight_single_row["model_name"],
            "macro_f1": best_lightweight_single_row["macro_f1"],
            "accuracy": best_lightweight_single_row["accuracy"],
            "prediction_time_seconds": best_lightweight_single_row["prediction_time_seconds"],
        }
    ]
)

display(single_model_choice_table)


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,CompactRF,Tree,Medium,Medium,0.8663,0.8663,0.8678,0.8663,0.8659,0.8659,0.8225,11.6511,0.055774,0.8552
1,DecisionTree,Tree,Light,Very high,0.8466,0.8466,0.8467,0.8466,0.8461,0.8461,0.7958,8.5457,0.021451,0.8294
2,LinearSVC,Linear,Very light,Very high,0.6718,0.6718,0.6722,0.6718,0.6703,0.6703,0.5634,699.3149,0.026041,0.6713
3,SGDLogLoss,Linear,Very light,Very high,0.6666,0.6666,0.6666,0.6666,0.6644,0.6644,0.5566,19.3367,0.024058,0.6671
4,LogisticRegression,Linear,Very light,Very high,0.6635,0.6635,0.6627,0.6635,0.6610,0.6610,0.5525,502.5065,0.025058,0.6636


,goal,model_name,macro_f1,accuracy,prediction_time_seconds
0,Best lightweight single model for later C conversion,DecisionTree,0.8461,0.8466,0.021451


## Step 41: Define The Five Hard-Voting Hybrid Models

We now build 5 hybrid candidates using the best tuned base models.

We use hard voting because:
- it is easy to explain
- it avoids probability calibration issues
- it is still easier to port into C logic than a soft-voting design


In [41]:
hybrid_model_specs = {
    "Hybrid_LinearVote": [
        ("logistic", best_single_estimators["LogisticRegression"]),
        ("sgd", best_single_estimators["SGDLogLoss"]),
        ("linearsvc", best_single_estimators["LinearSVC"]),
    ],
    "Hybrid_MixedLite": [
        ("logistic", best_single_estimators["LogisticRegression"]),
        ("sgd", best_single_estimators["SGDLogLoss"]),
        ("tree", best_single_estimators["DecisionTree"]),
    ],
    "Hybrid_TreeMix": [
        ("logistic", best_single_estimators["LogisticRegression"]),
        ("tree", best_single_estimators["DecisionTree"]),
        ("rf", best_single_estimators["CompactRF"]),
    ],
    "Hybrid_MarginTreeMix": [
        ("linearsvc", best_single_estimators["LinearSVC"]),
        ("tree", best_single_estimators["DecisionTree"]),
        ("rf", best_single_estimators["CompactRF"]),
    ],
    "Hybrid_AllFiveVote": [
        ("logistic", best_single_estimators["LogisticRegression"]),
        ("sgd", best_single_estimators["SGDLogLoss"]),
        ("linearsvc", best_single_estimators["LinearSVC"]),
        ("tree", best_single_estimators["DecisionTree"]),
        ("rf", best_single_estimators["CompactRF"]),
    ],
}

estimator_alias_to_single_name = {
    "logistic": "LogisticRegression",
    "sgd": "SGDLogLoss",
    "linearsvc": "LinearSVC",
    "tree": "DecisionTree",
    "rf": "CompactRF",
}

hybrid_catalog = pd.DataFrame(
    [
        {"hybrid_name": hybrid_name, "member_count": len(members), "members": ", ".join(name for name, _ in members)}
        for hybrid_name, members in hybrid_model_specs.items()
    ]
)

display(hybrid_catalog)


,hybrid_name,member_count,members
0,Hybrid_LinearVote,3,"logistic, sgd, linearsvc"
1,Hybrid_MixedLite,3,"logistic, sgd, tree"
2,Hybrid_TreeMix,3,"logistic, tree, rf"
3,Hybrid_MarginTreeMix,3,"linearsvc, tree, rf"
4,Hybrid_AllFiveVote,5,"logistic, sgd, linearsvc, tree, rf"


## Step 42: Create Empty Tracking Tables For The Hybrid Models

We use separate containers again so each hybrid can run in its own cell and show its own result clearly.


In [42]:
hybrid_model_rows = []
hybrid_model_predictions = {}


## Step 43: Evaluate Hybrid_LinearVote

This hybrid asks the three linear-style models to vote together.

This cell reuses the already tuned single models.
It does not retrain them from zero.

We combine their test predictions with hard voting so the logic stays easier to explain and easier to port later.


In [43]:
current_hybrid_name = "Hybrid_LinearVote"
current_estimators = hybrid_model_specs[current_hybrid_name]

current_member_table = pd.DataFrame(
    [
        {
            "hybrid_name": current_hybrid_name,
            "member_alias": alias_name,
            "base_model": estimator_alias_to_single_name[alias_name],
        }
        for alias_name, _ in current_estimators
    ]
)

fit_time = sum(
    single_model_fit_times[estimator_alias_to_single_name[alias_name]]
    for alias_name, _ in current_estimators
)

predict_start = time.perf_counter()
member_predictions = np.column_stack(
    [estimator.predict(X_test) for _, estimator in current_estimators]
)
current_y_pred = pd.DataFrame(member_predictions).mode(axis=1)[0].to_numpy()
prediction_time = time.perf_counter() - predict_start

current_hybrid_result_row = {
    **compute_metric_row(
        model_name=current_hybrid_name,
        model_family="Hard-voting hybrid",
        resource_class="Depends on members",
        c_export_friendliness="Medium",
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": np.nan,
}

hybrid_model_rows.append(current_hybrid_result_row)
hybrid_model_predictions[current_hybrid_name] = current_y_pred

print(f"Finished {current_hybrid_name}.")
display(current_member_table)
display(pd.DataFrame([current_hybrid_result_row]))


Finished Hybrid_LinearVote.


,hybrid_name,member_alias,base_model
0,Hybrid_LinearVote,logistic,LogisticRegression
1,Hybrid_LinearVote,sgd,SGDLogLoss
2,Hybrid_LinearVote,linearsvc,LinearSVC


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_LinearVote,Hard-voting hybrid,Depends on members,Medium,0.6674,0.6674,0.6672,0.6674,0.6653,0.6653,0.5576,1221.158,0.916609,NaN


## Step 44: Evaluate Hybrid_MixedLite

This hybrid mixes two linear models with one tree so you can see whether a small amount of tree structure helps.

This cell reuses the already tuned single models.
It does not retrain them from zero.

We combine their test predictions with hard voting so the logic stays easier to explain and easier to port later.


In [44]:
current_hybrid_name = "Hybrid_MixedLite"
current_estimators = hybrid_model_specs[current_hybrid_name]

current_member_table = pd.DataFrame(
    [
        {
            "hybrid_name": current_hybrid_name,
            "member_alias": alias_name,
            "base_model": estimator_alias_to_single_name[alias_name],
        }
        for alias_name, _ in current_estimators
    ]
)

fit_time = sum(
    single_model_fit_times[estimator_alias_to_single_name[alias_name]]
    for alias_name, _ in current_estimators
)

predict_start = time.perf_counter()
member_predictions = np.column_stack(
    [estimator.predict(X_test) for _, estimator in current_estimators]
)
current_y_pred = pd.DataFrame(member_predictions).mode(axis=1)[0].to_numpy()
prediction_time = time.perf_counter() - predict_start

current_hybrid_result_row = {
    **compute_metric_row(
        model_name=current_hybrid_name,
        model_family="Hard-voting hybrid",
        resource_class="Depends on members",
        c_export_friendliness="Medium",
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": np.nan,
}

hybrid_model_rows.append(current_hybrid_result_row)
hybrid_model_predictions[current_hybrid_name] = current_y_pred

print(f"Finished {current_hybrid_name}.")
display(current_member_table)
display(pd.DataFrame([current_hybrid_result_row]))


Finished Hybrid_MixedLite.


,hybrid_name,member_alias,base_model
0,Hybrid_MixedLite,logistic,LogisticRegression
1,Hybrid_MixedLite,sgd,SGDLogLoss
2,Hybrid_MixedLite,tree,DecisionTree


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_MixedLite,Hard-voting hybrid,Depends on members,Medium,0.6788,0.6788,0.6783,0.6788,0.6766,0.6766,0.5727,530.3889,0.845479,NaN


## Step 45: Evaluate Hybrid_TreeMix

This hybrid combines one linear model with two tree-based models.

This cell reuses the already tuned single models.
It does not retrain them from zero.

We combine their test predictions with hard voting so the logic stays easier to explain and easier to port later.


In [45]:
current_hybrid_name = "Hybrid_TreeMix"
current_estimators = hybrid_model_specs[current_hybrid_name]

current_member_table = pd.DataFrame(
    [
        {
            "hybrid_name": current_hybrid_name,
            "member_alias": alias_name,
            "base_model": estimator_alias_to_single_name[alias_name],
        }
        for alias_name, _ in current_estimators
    ]
)

fit_time = sum(
    single_model_fit_times[estimator_alias_to_single_name[alias_name]]
    for alias_name, _ in current_estimators
)

predict_start = time.perf_counter()
member_predictions = np.column_stack(
    [estimator.predict(X_test) for _, estimator in current_estimators]
)
current_y_pred = pd.DataFrame(member_predictions).mode(axis=1)[0].to_numpy()
prediction_time = time.perf_counter() - predict_start

current_hybrid_result_row = {
    **compute_metric_row(
        model_name=current_hybrid_name,
        model_family="Hard-voting hybrid",
        resource_class="Depends on members",
        c_export_friendliness="Medium",
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": np.nan,
}

hybrid_model_rows.append(current_hybrid_result_row)
hybrid_model_predictions[current_hybrid_name] = current_y_pred

print(f"Finished {current_hybrid_name}.")
display(current_member_table)
display(pd.DataFrame([current_hybrid_result_row]))


Finished Hybrid_TreeMix.


,hybrid_name,member_alias,base_model
0,Hybrid_TreeMix,logistic,LogisticRegression
1,Hybrid_TreeMix,tree,DecisionTree
2,Hybrid_TreeMix,rf,CompactRF


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_TreeMix,Hard-voting hybrid,Depends on members,Medium,0.854,0.8539,0.8543,0.8539,0.8532,0.8532,0.8059,522.7034,0.832425,NaN


## Step 46: Evaluate Hybrid_MarginTreeMix

This hybrid uses the strongest margin-based linear model with both tree-based models.

This cell reuses the already tuned single models.
It does not retrain them from zero.

We combine their test predictions with hard voting so the logic stays easier to explain and easier to port later.


In [46]:
current_hybrid_name = "Hybrid_MarginTreeMix"
current_estimators = hybrid_model_specs[current_hybrid_name]

current_member_table = pd.DataFrame(
    [
        {
            "hybrid_name": current_hybrid_name,
            "member_alias": alias_name,
            "base_model": estimator_alias_to_single_name[alias_name],
        }
        for alias_name, _ in current_estimators
    ]
)

fit_time = sum(
    single_model_fit_times[estimator_alias_to_single_name[alias_name]]
    for alias_name, _ in current_estimators
)

predict_start = time.perf_counter()
member_predictions = np.column_stack(
    [estimator.predict(X_test) for _, estimator in current_estimators]
)
current_y_pred = pd.DataFrame(member_predictions).mode(axis=1)[0].to_numpy()
prediction_time = time.perf_counter() - predict_start

current_hybrid_result_row = {
    **compute_metric_row(
        model_name=current_hybrid_name,
        model_family="Hard-voting hybrid",
        resource_class="Depends on members",
        c_export_friendliness="Medium",
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": np.nan,
}

hybrid_model_rows.append(current_hybrid_result_row)
hybrid_model_predictions[current_hybrid_name] = current_y_pred

print(f"Finished {current_hybrid_name}.")
display(current_member_table)
display(pd.DataFrame([current_hybrid_result_row]))


Finished Hybrid_MarginTreeMix.


,hybrid_name,member_alias,base_model
0,Hybrid_MarginTreeMix,linearsvc,LinearSVC
1,Hybrid_MarginTreeMix,tree,DecisionTree
2,Hybrid_MarginTreeMix,rf,CompactRF


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_MarginTreeMix,Hard-voting hybrid,Depends on members,Medium,0.8551,0.8551,0.8557,0.8551,0.8544,0.8544,0.8074,719.5117,0.878772,NaN


## Step 47: Evaluate Hybrid_AllFiveVote

This is the broadest vote because it lets all five single models participate.

This cell reuses the already tuned single models.
It does not retrain them from zero.

We combine their test predictions with hard voting so the logic stays easier to explain and easier to port later.


In [47]:
current_hybrid_name = "Hybrid_AllFiveVote"
current_estimators = hybrid_model_specs[current_hybrid_name]

current_member_table = pd.DataFrame(
    [
        {
            "hybrid_name": current_hybrid_name,
            "member_alias": alias_name,
            "base_model": estimator_alias_to_single_name[alias_name],
        }
        for alias_name, _ in current_estimators
    ]
)

fit_time = sum(
    single_model_fit_times[estimator_alias_to_single_name[alias_name]]
    for alias_name, _ in current_estimators
)

predict_start = time.perf_counter()
member_predictions = np.column_stack(
    [estimator.predict(X_test) for _, estimator in current_estimators]
)
current_y_pred = pd.DataFrame(member_predictions).mode(axis=1)[0].to_numpy()
prediction_time = time.perf_counter() - predict_start

current_hybrid_result_row = {
    **compute_metric_row(
        model_name=current_hybrid_name,
        model_family="Hard-voting hybrid",
        resource_class="Depends on members",
        c_export_friendliness="Medium",
        y_true=y_test,
        y_pred=current_y_pred,
        fit_time_seconds=fit_time,
        prediction_time_seconds=prediction_time,
    ),
    "best_cv_macro_f1": np.nan,
}

hybrid_model_rows.append(current_hybrid_result_row)
hybrid_model_predictions[current_hybrid_name] = current_y_pred

print(f"Finished {current_hybrid_name}.")
display(current_member_table)
display(pd.DataFrame([current_hybrid_result_row]))


Finished Hybrid_AllFiveVote.


,hybrid_name,member_alias,base_model
0,Hybrid_AllFiveVote,logistic,LogisticRegression
1,Hybrid_AllFiveVote,sgd,SGDLogLoss
2,Hybrid_AllFiveVote,linearsvc,LinearSVC
3,Hybrid_AllFiveVote,tree,DecisionTree
4,Hybrid_AllFiveVote,rf,CompactRF


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_AllFiveVote,Hard-voting hybrid,Depends on members,Medium,0.6997,0.6997,0.6987,0.6997,0.6976,0.6976,0.6006,1241.3549,0.892327,NaN


## Step 48: Rank The Five Hybrid Models

We use the same ranking rule:
1. test macro F1
2. test accuracy
3. lower prediction time


In [48]:
hybrid_model_results = pd.DataFrame(hybrid_model_rows)
hybrid_model_results_ranked = hybrid_model_results.sort_values(
    ["macro_f1", "accuracy", "prediction_time_seconds"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(hybrid_model_results)
display(hybrid_model_results_ranked)


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_LinearVote,Hard-voting hybrid,Depends on members,Medium,0.6674,0.6674,0.6672,0.6674,0.6653,0.6653,0.5576,1221.1580,0.916609,NaN
1,Hybrid_MixedLite,Hard-voting hybrid,Depends on members,Medium,0.6788,0.6788,0.6783,0.6788,0.6766,0.6766,0.5727,530.3889,0.845479,NaN
2,Hybrid_TreeMix,Hard-voting hybrid,Depends on members,Medium,0.8540,0.8539,0.8543,0.8539,0.8532,0.8532,0.8059,522.7034,0.832425,NaN
3,Hybrid_MarginTreeMix,Hard-voting hybrid,Depends on members,Medium,0.8551,0.8551,0.8557,0.8551,0.8544,0.8544,0.8074,719.5117,0.878772,NaN
4,Hybrid_AllFiveVote,Hard-voting hybrid,Depends on members,Medium,0.6997,0.6997,0.6987,0.6997,0.6976,0.6976,0.6006,1241.3549,0.892327,NaN


,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Hybrid_MarginTreeMix,Hard-voting hybrid,Depends on members,Medium,0.8551,0.8551,0.8557,0.8551,0.8544,0.8544,0.8074,719.5117,0.878772,NaN
1,Hybrid_TreeMix,Hard-voting hybrid,Depends on members,Medium,0.8540,0.8539,0.8543,0.8539,0.8532,0.8532,0.8059,522.7034,0.832425,NaN
2,Hybrid_AllFiveVote,Hard-voting hybrid,Depends on members,Medium,0.6997,0.6997,0.6987,0.6997,0.6976,0.6976,0.6006,1241.3549,0.892327,NaN
3,Hybrid_MixedLite,Hard-voting hybrid,Depends on members,Medium,0.6788,0.6788,0.6783,0.6788,0.6766,0.6766,0.5727,530.3889,0.845479,NaN
4,Hybrid_LinearVote,Hard-voting hybrid,Depends on members,Medium,0.6674,0.6674,0.6672,0.6674,0.6653,0.6653,0.5576,1221.1580,0.916609,NaN


## Step 49: Build The Final Leaderboard And Final Recommendations

We now combine the single-model and hybrid-model tables.

Final choices:
- one best lightweight single model
- one best hybrid model


In [49]:
single_model_results_with_type = single_model_results.copy()
single_model_results_with_type.insert(0, "model_group", "Single")

hybrid_model_results_with_type = hybrid_model_results.copy()
hybrid_model_results_with_type.insert(0, "model_group", "Hybrid")

final_leaderboard = pd.concat(
    [single_model_results_with_type, hybrid_model_results_with_type],
    ignore_index=True,
).sort_values(
    ["macro_f1", "accuracy", "prediction_time_seconds"],
    ascending=[False, False, True],
).reset_index(drop=True)

best_hybrid_row = hybrid_model_results_ranked.iloc[0]

final_recommendation_table = pd.DataFrame(
    [
        {
            "goal": "Best lightweight single model for later C conversion",
            "recommended_model": best_lightweight_single_row["model_name"],
            "macro_f1": best_lightweight_single_row["macro_f1"],
            "accuracy": best_lightweight_single_row["accuracy"],
            "prediction_time_seconds": best_lightweight_single_row["prediction_time_seconds"],
        },
        {
            "goal": "Best hybrid model for later C conversion",
            "recommended_model": best_hybrid_row["model_name"],
            "macro_f1": best_hybrid_row["macro_f1"],
            "accuracy": best_hybrid_row["accuracy"],
            "prediction_time_seconds": best_hybrid_row["prediction_time_seconds"],
        },
    ]
)

display(final_leaderboard)
display(final_recommendation_table)

best_lightweight_single_name = best_lightweight_single_row["model_name"]
best_hybrid_name = best_hybrid_row["model_name"]


,model_group,model_name,model_family,resource_class,c_export_friendliness,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,fit_time_seconds,prediction_time_seconds,best_cv_macro_f1
0,Single,CompactRF,Tree,Medium,Medium,0.8663,0.8663,0.8678,0.8663,0.8659,0.8659,0.8225,11.6511,0.055774,0.8552
1,Hybrid,Hybrid_MarginTreeMix,Hard-voting hybrid,Depends on members,Medium,0.8551,0.8551,0.8557,0.8551,0.8544,0.8544,0.8074,719.5117,0.878772,NaN
2,Hybrid,Hybrid_TreeMix,Hard-voting hybrid,Depends on members,Medium,0.8540,0.8539,0.8543,0.8539,0.8532,0.8532,0.8059,522.7034,0.832425,NaN
3,Single,DecisionTree,Tree,Light,Very high,0.8466,0.8466,0.8467,0.8466,0.8461,0.8461,0.7958,8.5457,0.021451,0.8294
4,Hybrid,Hybrid_AllFiveVote,Hard-voting hybrid,Depends on members,Medium,0.6997,0.6997,0.6987,0.6997,0.6976,0.6976,0.6006,1241.3549,0.892327,NaN
5,Hybrid,Hybrid_MixedLite,Hard-voting hybrid,Depends on members,Medium,0.6788,0.6788,0.6783,0.6788,0.6766,0.6766,0.5727,530.3889,0.845479,NaN
6,Single,LinearSVC,Linear,Very light,Very high,0.6718,0.6718,0.6722,0.6718,0.6703,0.6703,0.5634,699.3149,0.026041,0.6713
7,Hybrid,Hybrid_LinearVote,Hard-voting hybrid,Depends on members,Medium,0.6674,0.6674,0.6672,0.6674,0.6653,0.6653,0.5576,1221.1580,0.916609,NaN
8,Single,SGDLogLoss,Linear,Very light,Very high,0.6666,0.6666,0.6666,0.6666,0.6644,0.6644,0.5566,19.3367,0.024058,0.6671
9,Single,LogisticRegression,Linear,Very light,Very high,0.6635,0.6635,0.6627,0.6635,0.6610,0.6610,0.5525,502.5065,0.025058,0.6636


,goal,recommended_model,macro_f1,accuracy,prediction_time_seconds
0,Best lightweight single model for later C conversion,DecisionTree,0.8461,0.8466,0.021451
1,Best hybrid model for later C conversion,Hybrid_MarginTreeMix,0.8544,0.8551,0.878772


## Step 50: Show The Confusion Matrix And Class Report For The Best Lightweight Single Model

This is the main lightweight candidate for later C conversion.


In [50]:
best_lightweight_predictions = single_model_predictions[best_lightweight_single_name]

best_lightweight_confusion = pd.DataFrame(
    confusion_matrix(y_test, best_lightweight_predictions, labels=class_labels),
    index=[f"true_{label}" for label in class_labels],
    columns=[f"pred_{label}" for label in class_labels],
)

best_lightweight_report = pd.DataFrame(
    classification_report(
        y_test,
        best_lightweight_predictions,
        output_dict=True,
        zero_division=0,
    )
).T

best_lightweight_one_vs_rest = build_one_vs_rest_table(
    best_lightweight_single_name,
    y_test,
    best_lightweight_predictions,
    class_labels,
)

print(f"Best lightweight single model: {best_lightweight_single_name}")
display(best_lightweight_confusion)
display(best_lightweight_report)
display(best_lightweight_one_vs_rest)


Best lightweight single model: DecisionTree


,pred_Adult,pred_Game,pred_Payment,pred_Social-Media
true_Adult,1535,102,43,123
true_Game,82,1591,33,96
true_Payment,52,58,1610,83
true_Social-Media,176,170,88,1368


,precision,recall,f1-score,support
Adult,0.831978,0.851359,0.841557,1803.000000
Game,0.828214,0.882908,0.854687,1802.000000
Payment,0.907554,0.892956,0.900196,1803.000000
Social-Media,0.819162,0.759156,0.788018,1802.000000
accuracy,0.846602,0.846602,0.846602,0.846602
macro avg,0.846727,0.846595,0.846115,7210.000000
weighted avg,0.846733,0.846602,0.846121,7210.000000


,model_name,class_label,tp,tn,fp,fn,specificity,false_positive_rate,false_negative_rate,negative_predictive_value
0,DecisionTree,Adult,1535,5097,310,268,0.9427,0.0573,0.1486,0.9500
1,DecisionTree,Game,1591,5078,330,211,0.9390,0.0610,0.1171,0.9601
2,DecisionTree,Payment,1610,5243,164,193,0.9697,0.0303,0.1070,0.9645
3,DecisionTree,Social-Media,1368,5106,302,434,0.9442,0.0558,0.2408,0.9217


## Step 51: Show The Confusion Matrix And Class Report For The Best Hybrid Model

This is the strongest hybrid candidate from the current notebook design.


In [51]:
best_hybrid_predictions = hybrid_model_predictions[best_hybrid_name]

best_hybrid_confusion = pd.DataFrame(
    confusion_matrix(y_test, best_hybrid_predictions, labels=class_labels),
    index=[f"true_{label}" for label in class_labels],
    columns=[f"pred_{label}" for label in class_labels],
)

best_hybrid_report = pd.DataFrame(
    classification_report(
        y_test,
        best_hybrid_predictions,
        output_dict=True,
        zero_division=0,
    )
).T

best_hybrid_one_vs_rest = build_one_vs_rest_table(
    best_hybrid_name,
    y_test,
    best_hybrid_predictions,
    class_labels,
)

print(f"Best hybrid model: {best_hybrid_name}")
display(best_hybrid_confusion)
display(best_hybrid_report)
display(best_hybrid_one_vs_rest)


Best hybrid model: Hybrid_MarginTreeMix


,pred_Adult,pred_Game,pred_Payment,pred_Social-Media
true_Adult,1525,103,38,137
true_Game,85,1623,33,61
true_Payment,68,62,1636,37
true_Social-Media,155,171,95,1381


,precision,recall,f1-score,support
Adult,0.831969,0.845813,0.838834,1803.000000
Game,0.828484,0.900666,0.863068,1802.000000
Payment,0.907880,0.907377,0.907628,1803.000000
Social-Media,0.854579,0.766371,0.808075,1802.000000
accuracy,0.855062,0.855062,0.855062,0.855062
macro avg,0.855728,0.855056,0.854401,7210.000000
weighted avg,0.855732,0.855062,0.854407,7210.000000


,model_name,class_label,tp,tn,fp,fn,specificity,false_positive_rate,false_negative_rate,negative_predictive_value
0,Hybrid_MarginTreeMix,Adult,1525,5099,308,278,0.9430,0.0570,0.1542,0.9483
1,Hybrid_MarginTreeMix,Game,1623,5072,336,179,0.9379,0.0621,0.0993,0.9659
2,Hybrid_MarginTreeMix,Payment,1636,5241,166,167,0.9693,0.0307,0.0926,0.9691
3,Hybrid_MarginTreeMix,Social-Media,1381,5173,235,421,0.9565,0.0435,0.2336,0.9247


## Step 52: Write The Final Beginner-Friendly Interpretation

How to read the final result:
- the best lightweight single model is the first model to try for C conversion
- the best hybrid model is the accuracy-oriented comparison target
- if the hybrid only improves a little, the lightweight single model may still be the better deployment choice
- if accuracy is still below your target, the next likely improvements are better features, cleaner labels, or a different balancing strategy


In [52]:
final_interpretation = pd.DataFrame(
    [
        {
            "question": "What is the first model to try converting into C?",
            "answer": best_lightweight_single_name,
        },
        {
            "question": "What is the best hybrid comparison model?",
            "answer": best_hybrid_name,
        },
        {
            "question": "What is the most important research score in this notebook?",
            "answer": "Test macro F1",
        },
        {
            "question": "What is the main warning about these results?",
            "answer": "The dataset was category-balanced before the 80:20 split, so this is a controlled research comparison rather than a strict raw-data deployment benchmark.",
        },
    ]
)

display(final_interpretation)


,question,answer
0,What is the first model to try converting into C?,DecisionTree
1,What is the best hybrid comparison model?,Hybrid_MarginTreeMix
2,What is the most important research score in this notebook?,Test macro F1
3,What is the main warning about these results?,"The dataset was category-balanced before the 80:20 split, so this is a contr..."
